### Catastrophe Hist → Lakebase (DevConnect)

Syncs `bronze_hist_orders` and `bronze_hist_refunds` from UC Delta into
the shared Lakebase `databricks_postgres` database so Lakebase SQL queries 2–6 in
`demos/devconnect-runbooks/2-lakebase-issue-fair-refund.sql` can join live `orders` to history.

- UC sources:     `{CATALOG}.orders.bronze_hist_*`
- UC synced entries: `{CATALOG}.lakebase.bronze_hist_*`
- Postgres tables:   `databricks_postgres.public.bronze_hist_*` (same names)

Depends on `Catastrophe_History` (source tables exist) and `Lakebase_Project`.

In [ ]:
%pip install --upgrade "databricks-sdk>=0.81.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re
import sys

sys.path.append('../utils')
from uc_state import add
from lakebase_autoscale import (
    get_or_create_autoscale_synced_table,
    get_autoscale_synced_table,
)
import status

CATALOG = dbutils.widgets.get("CATALOG")
try:
    ORDERS_SCHEMA = dbutils.widgets.get("ORDERS_SCHEMA") or "orders"
except Exception:
    ORDERS_SCHEMA = "orders"

PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-caspers".lower())
BRANCH_PATH = f"projects/{PROJECT_ID}/branches/production"
POSTGRES_DB = "databricks_postgres"
SYNC_SCHEMA = "lakebase"

SYNC_SPECS = [
    {
        "basename": "bronze_hist_orders",
        "source": f"{CATALOG}.{ORDERS_SCHEMA}.bronze_hist_orders",
        "primary_key": ["order_id"],
    },
    {
        "basename": "bronze_hist_refunds",
        "source": f"{CATALOG}.{ORDERS_SCHEMA}.bronze_hist_refunds",
        "primary_key": ["order_id"],
    },
]

print(f"CATALOG      = {CATALOG}")
print(f"ORDERS_SCHEMA = {ORDERS_SCHEMA}")
print(f"PROJECT_ID   = {PROJECT_ID}")
print(f"POSTGRES_DB  = {POSTGRES_DB}")

In [ ]:
def _enable_cdf(table_name: str) -> None:
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()
        enabled = any(
            row.key == "delta.enableChangeDataFeed" and str(row.value).lower() == "true"
            for row in props
        )
    except Exception:
        enabled = False
    if enabled:
        print(f"  CDF already enabled on {table_name}")
        return
    spark.sql(
        f"ALTER TABLE {table_name} SET TBLPROPERTIES "
        f"('delta.enableChangeDataFeed' = 'true')"
    )
    print(f"  ✅ CDF enabled on {table_name}")

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SYNC_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}._lakebase_storage")

for spec in SYNC_SPECS:
    basename = spec["basename"]
    source = spec["source"]
    synced = f"{CATALOG}.{SYNC_SCHEMA}.{basename}"
    backing = f"{CATALOG}._lakebase_storage.{basename}"

    if not spark.catalog.tableExists(source):
        raise RuntimeError(f"Source table missing: {source} (run Catastrophe_History first)")

    print(f"\n=== {basename} ===")
    _enable_cdf(source)

    existing = get_autoscale_synced_table(w, synced)
    if existing is not None:
        status.reuse(f"Found existing synced table: {synced}")
        continue

    for stale in (synced, backing):
        try:
            spark.sql(f"DROP TABLE IF EXISTS {stale}")
            status.drop(f"Cleared stale Delta at {stale}")
        except Exception as e:
            status.warn(f"Could not drop {stale}: {e}")

    get_or_create_autoscale_synced_table(
        w,
        synced_table_name=synced,
        source_table_full_name=source,
        primary_key_columns=spec["primary_key"],
        branch=BRANCH_PATH,
        postgres_database=POSTGRES_DB,
        scheduling_policy="CONTINUOUS",
        create_database_objects_if_missing=True,
    )
    status.ok(f"Created synced table {synced} → {POSTGRES_DB}.public.{basename}")
    add(CATALOG, "autoscale_synced_tables", {
        "synced_table_name": synced,
        "project_id": PROJECT_ID,
        "postgres_database": POSTGRES_DB,
        "postgres_schema": "public",
        "source_table": source,
    })

print("\n✅ Catastrophe hist Lakebase sync complete")